# OLD Monte Carlo downstream synaptic fields (networkx walk) — for comparison

This mirrors `downstream_plotting.ipynb` but loads the **old Monte Carlo** downstream fields produced by `compute_downstream_mc_sim.py` (the legacy `Paths.monte_carlo` networkx walk, 100k reps, `get_synaptic_fields`).

Fields are cached per neuron in `fields_ds_mcsim_<side>/<LC>_mcsim.npz`, keyed by input channel (+ an OR-combined `all`), a single frame per channel (already aggregated over the LC type's cells). The same `field_plotting` helpers work on them.

A dedicated cell compares these MC fields against the analytic fields (`fields_ds_<side>/`), with an L1 focus.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import glob
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn
import field_plotting
importlib.reload(field_plotting)
import field_plotting as fp
from flywire_tools.connectome import load_SynapticFields

SIDE = 'right'   # 'right' or 'left'

In [ ]:
# neuron ordering (LCe, LC, LPLC by class number), matching the analytic notebook
vnt = pd.read_csv('visual_neuron_types.csv')
lc_types = fp.neuron_types(vnt, 'LC')
lplc_types = fp.neuron_types(vnt, 'LPLC')
cell_classes = ['LCe', 'LC', 'LPLC']
import re
def _order_key(n):
    for i, c in enumerate(cell_classes):
        if n.startswith(c):
            m = re.search(r'\d+', n)
            return (i, int(m.group()) if m else 0)
    return (len(cell_classes), 0)
neurons = sorted(set(lc_types) | set(lplc_types), key=_order_key)

## Load the old-MC downstream fields

In [ ]:
cache_dir = f'fields_ds_mcsim_{SIDE}'
all_fields = {}
for neuron in neurons:
    fn = os.path.join(cache_dir, f'{neuron}_mcsim.npz')
    if os.path.exists(fn):
        all_fields[neuron] = load_SynapticFields(fn)
plotted = [n for n in neurons if n in all_fields]
print(f'loaded {len(plotted)} / {len(neurons)} neurons from {cache_dir}')
plotted

In [ ]:
fig, axes = fp.plot_field_grid(
    all_fields, plotted,
    out_png=f'mcsim_downstream_fields_grid_{SIDE}.png',
    title=f'OLD Monte Carlo downstream synaptic fields onto LC/LPLC ({SIDE} eye)')
plt.show()

## Analytic vs old-MC comparison (L1 focus)

Loads the analytic fields from `fields_ds_<side>/` and compares, per input channel, the aggregate (over targets) reach against the MC fields: Pearson correlation over the union support and the ratio of total mass. Large per-channel disagreement (especially for **L1**) localizes where the two methods diverge.

In [ ]:
an_dir = f'fields_ds_{SIDE}'
channels = fp.STARTING_POINTS + ['all']
rows = {}
for neuron in plotted:
    an_fn = os.path.join(an_dir, f'{neuron}_analytic.npz')
    if not os.path.exists(an_fn):
        continue
    an = load_SynapticFields(an_fn)
    mc = all_fields[neuron]
    for ch in channels:
        if ch not in an or ch not in mc:
            continue
        a = fp.aggregate_image(an[ch]).ravel()
        m = fp.aggregate_image(mc[ch]).ravel()
        keep = (a > 0) | (m > 0)
        r = (np.corrcoef(a[keep], m[keep])[0, 1]
             if keep.sum() > 2 and a[keep].std() > 0 and m[keep].std() > 0 else np.nan)
        rows.setdefault(neuron, {})[ch] = r
corr_an_mc = pd.DataFrame(rows).T[channels]
print('analytic-vs-MC Pearson correlation per neuron x channel')
corr_an_mc.round(3)

In [ ]:
# heatmap of analytic-vs-MC agreement; low values (esp. L1 column) = where methods diverge
fig, ax = plt.subplots(figsize=(6, 0.28 * len(corr_an_mc) + 2), constrained_layout=True)
sbn.heatmap(corr_an_mc, ax=ax, cmap='rocket', vmin=0, vmax=1,
            cbar_kws={'label': 'analytic vs MC correlation'})
ax.set_title(f'Analytic vs old-MC agreement (side={SIDE})')
ax.set_xlabel('input channel')
ax.set_ylabel('neuron')
fig.savefig(f'analytic_vs_mcsim_corr_{SIDE}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# side-by-side L1 field for one neuron: analytic vs old-MC
demo = 'LC11' if 'LC11' in plotted else plotted[0]
ch = 'L1'
an = load_SynapticFields(os.path.join(an_dir, f'{demo}_analytic.npz'))
mc = all_fields[demo]
from matplotlib.colors import Normalize
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
for ax, flds, ttl in zip(axs, [an, mc], [f'{demo} {ch} analytic', f'{demo} {ch} old-MC']):
    img = fp.aggregate_image(flds[ch])
    f2 = type(flds[ch])(img[None], ps=flds[ch].ps, qs=flds[ch].qs)
    vmax = max(float(fp.aggregate_image(an[ch]).max()), float(fp.aggregate_image(mc[ch]).max())) or 1.0
    fp.draw_hex(ax, f2, norm=Normalize(0, vmax), axis_lim=90)
    ax.set_title(ttl, fontsize=10)
plt.show()

## Phase 1–3 on the MC fields

The same sensitivity / input-pattern / taxonomy analyses as the analytic notebook, computed on the MC fields for direct comparison.

In [ ]:
sens_summary = fp.sensitivity_summary(all_fields, plotted, reps=10000, confidence=95)
sens_summary.round(4)

In [ ]:
gnorm = fp.gain_normalized_summary(all_fields, plotted, sens_summary)
resid_df, corr_df = fp.combined_scaling_check(all_fields, plotted)
fig, axes = plt.subplots(1, 2, figsize=(11, 0.28 * len(plotted) + 2), constrained_layout=True)
sbn.heatmap(gnorm, ax=axes[0], cmap='magma', cbar_kws={'label': 'gain-normalized mean reach'})
axes[0].set_title('MC input-channel pattern (gain-normalized)')
axes[0].set_xlabel('input channel'); axes[0].set_ylabel('neuron')
sbn.heatmap(resid_df, ax=axes[1], cmap='rocket_r', vmin=0, cbar_kws={'label': 'residual fraction'})
axes[1].set_title('MC deviation from scaled combined')
axes[1].set_xlabel('input channel')
plt.show()

In [ ]:
X, keep = fp.rf_matrix(all_fields, plotted, sens_summary, normalize='gain')
scores, s, ev, k = fp.svd_reduce(X, var_threshold=0.90)
print(f'RF matrix {X.shape}; kept {k} components for >=90% variance')
lab_hdb = fp.cluster_rfs(scores, method='hdbscan', min_cluster_size=2)
fig, axes = fp.plot_rf_taxonomy(all_fields, plotted, lab_hdb)
fig.suptitle(f'MC receptive-field taxonomy by cluster (side={SIDE})', y=1.002)
plt.show()